# RAG Fundamentals — Session Notebook
### AIM Program | IIT Patna × Masai School

---

**Topic:** Retrieval-Augmented Generation (RAG) Fundamentals  
**Estimated Time:** 90–110 minutes  
**Difficulty:** Intermediate  

---

## What You Will Build

By the end of this notebook, you will have built a **complete RAG pipeline from scratch**:

1. Load and inspect a sample document corpus
2. Implement **three chunking strategies**: fixed-size, overlapping, section-based
3. Embed chunks using a **sentence transformer** model
4. Build a **FAISS vector index** for fast semantic retrieval
5. Retrieve relevant chunks for a query and **build a grounded prompt**
6. Implement a **grounding check** to verify answer faithfulness
7. Compare chunking strategies on retrieval quality

No external API key is required — we use open-source models throughout.

---

## Learning Goals

- LO1: Explain the RAG architecture — retriever and generator roles
- LO2: Implement and compare three chunking strategies
- LO3: Build an end-to-end retrieval pipeline with sentence embeddings and FAISS
- LO4: Write and interpret a grounding check for answer faithfulness

---
## Section 0: Environment Setup

Install required libraries. This cell only needs to run once.

In [ ]:
# Install dependencies
!pip install -q sentence-transformers faiss-cpu numpy

In [ ]:
import re
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from collections import Counter

print("Libraries loaded successfully.")
print(f"FAISS version: {faiss.__version__}")

---
## Section 1: Dataset Introduction

We will work with a **synthetic clinical knowledge base** — a collection of short medical articles about common conditions and medications. This simulates a real-world scenario where an organization wants a QA bot over its internal documents.

The corpus has been designed to have:
- Multiple documents covering different topics
- Natural section breaks (double newlines)
- Varying document lengths
- Overlapping terminology across documents (to make retrieval non-trivial)

In [ ]:
# Synthetic clinical knowledge base
DOCUMENTS = [
    """Hypertension Management Protocol

Hypertension, or high blood pressure, is defined as a sustained blood pressure reading above 130/80 mmHg. 
It is one of the leading risk factors for cardiovascular disease, stroke, and renal failure worldwide.

First-line Treatment

For patients with stage 1 hypertension (130-139/80-89 mmHg) without compelling comorbidities, 
lifestyle modifications are recommended as initial therapy. These include dietary sodium reduction 
to less than 2.3 grams per day, regular aerobic exercise for at least 150 minutes per week, 
weight reduction, and moderation of alcohol consumption.

Pharmacological Therapy

If lifestyle modifications are insufficient after 3 months, or for stage 2 hypertension, 
pharmacological therapy is initiated. First-line drug classes include ACE inhibitors (e.g., lisinopril), 
angiotensin receptor blockers (ARBs), calcium channel blockers (e.g., amlodipine), 
and thiazide diuretics. Combination therapy is often required to achieve target blood pressure.

Monitoring and Follow-up

Patients on antihypertensive therapy should be monitored monthly until blood pressure is controlled, 
then every 3-6 months. Renal function and electrolytes should be checked annually, 
or more frequently if ACE inhibitors or diuretics are used.""",

    """Ibuprofen: Clinical Reference Guide

Ibuprofen is a non-steroidal anti-inflammatory drug (NSAID) used for the relief of mild to moderate pain, 
fever, and inflammation. It works by inhibiting cyclooxygenase (COX-1 and COX-2) enzymes, 
reducing prostaglandin synthesis.

Dosage Guidelines

For adults, the standard dose is 200-400 mg every 4 to 6 hours as needed. 
The maximum daily dose is 1200 mg for over-the-counter use, and 3200 mg under medical supervision. 
Ibuprofen should be taken with food or milk to reduce gastrointestinal irritation.

Side Effects and Contraindications

Common side effects include gastrointestinal discomfort, nausea, and heartburn. 
Serious risks include peptic ulceration, gastrointestinal bleeding, and renal impairment. 
Ibuprofen is contraindicated in patients with active peptic ulcer disease, severe renal failure, 
and in the third trimester of pregnancy. Caution is required in elderly patients and those with 
a history of cardiovascular disease due to increased risk of myocardial infarction and stroke.

Drug Interactions

Ibuprofen may interact with anticoagulants such as warfarin, increasing bleeding risk. 
Concurrent use with other NSAIDs or aspirin increases gastrointestinal side effects. 
Ibuprofen can reduce the antihypertensive effect of ACE inhibitors and diuretics.""",

    """Type 2 Diabetes: Overview and Management

Type 2 diabetes mellitus is a chronic metabolic disorder characterized by insulin resistance 
and relative insulin deficiency, resulting in hyperglycemia. It accounts for approximately 
90-95% of all diabetes cases globally.

Diagnostic Criteria

Diagnosis is confirmed by any of the following: fasting plasma glucose of 126 mg/dL or higher, 
2-hour plasma glucose of 200 mg/dL or higher during an oral glucose tolerance test, 
HbA1c of 6.5% or higher, or a random plasma glucose of 200 mg/dL or higher with symptoms.

Treatment Approach

Metformin remains the first-line pharmacological agent for type 2 diabetes due to its efficacy, 
safety profile, and low cost. It reduces hepatic glucose production and improves insulin sensitivity. 
Second-line agents include SGLT-2 inhibitors (e.g., empagliflozin), GLP-1 receptor agonists 
(e.g., semaglutide), and DPP-4 inhibitors. Insulin therapy is initiated when glycemic targets 
are not achieved with oral agents.

Lifestyle and Monitoring

Dietary modification emphasizing reduced carbohydrate intake and increased fiber is central to management. 
HbA1c should be measured every 3 months until the target of less than 7% is achieved, 
then every 6 months. Annual screening for complications including nephropathy, retinopathy, 
and neuropathy is recommended.""",

    """Antibiotic Stewardship Guidelines

Antibiotic stewardship refers to a coordinated set of interventions designed to improve and measure 
the appropriate use of antibiotics. The goal is to promote the selection of the optimal antibiotic 
drug regimen, dose, duration, and route of administration.

Core Principles

Antibiotics should only be prescribed when there is clear evidence of bacterial infection. 
Culture and sensitivity testing should be performed before initiating therapy when possible. 
Broad-spectrum antibiotics should be de-escalated to narrow-spectrum agents once sensitivities 
are known. Duration of therapy should be as short as clinically effective.

Common Mistakes to Avoid

Prescribing antibiotics for viral infections such as the common cold, influenza, or most sore throats 
contributes to antimicrobial resistance without clinical benefit. Failing to adjust dose for renal 
or hepatic impairment leads to toxicity or treatment failure. Empirical therapy should be guided 
by local resistance patterns rather than habit.

Documentation Requirements

Every antibiotic prescription must include the indication, the anticipated duration of therapy, 
and a review date. This documentation supports antibiotic time-out reviews at 48-72 hours, 
where the clinical team reassesses whether antibiotic therapy remains appropriate."""
]

print(f"Corpus loaded: {len(DOCUMENTS)} documents")
for i, doc in enumerate(DOCUMENTS):
    print(f"  Doc {i+1}: {doc.split(chr(10))[0][:60]}... ({len(doc)} chars)")

---
## Section 2: Document Chunking Strategies

We implement all three chunking strategies. After each, we'll inspect the chunks produced from Document 2 (Ibuprofen reference guide) to compare quality.

**Key question to keep in mind as you run these:** Which strategy produces chunks that you'd actually want to retrieve if someone asked *"What are the side effects of ibuprofen?"*

### Strategy 1: Fixed-Size Chunking

In [ ]:
def chunk_fixed(text, chunk_size=300):
    """
    Split text into non-overlapping chunks of exactly chunk_size characters.
    Strips whitespace from each chunk and skips empty ones.
    """
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
    return chunks


# Test on Document 2
fixed_chunks = chunk_fixed(DOCUMENTS[1], chunk_size=300)

print(f"Fixed-size chunking produced {len(fixed_chunks)} chunks from Doc 2")
print("\n" + "="*70)
for i, chunk in enumerate(fixed_chunks):
    print(f"\n[Chunk {i+1}] ({len(chunk)} chars):")
    print(chunk)
    print("-"*50)

**Observation:** Notice how some chunks cut sentences mid-thought. What impact would this have on a retrieval system trying to answer *"What are the side effects of ibuprofen?"*

### Strategy 2: Overlapping Chunking

In [ ]:
def chunk_overlap(text, chunk_size=300, overlap=60):
    """
    Split text into chunks of chunk_size characters, 
    with 'overlap' characters shared between adjacent chunks.
    Step size = chunk_size - overlap.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += (chunk_size - overlap)  # Slide forward by (chunk_size - overlap)
    return chunks


# Test on Document 2
overlap_chunks = chunk_overlap(DOCUMENTS[1], chunk_size=300, overlap=60)

print(f"Overlapping chunking produced {len(overlap_chunks)} chunks from Doc 2")
print(f"(vs {len(fixed_chunks)} for fixed-size — {len(overlap_chunks) - len(fixed_chunks)} extra chunks due to overlap)")
print("\n" + "="*70)
for i, chunk in enumerate(overlap_chunks[:3]):  # Show first 3 chunks
    print(f"\n[Chunk {i+1}] ({len(chunk)} chars):")
    print(chunk)
    print("-"*50)

# Visualize the overlap between chunk 1 and chunk 2
print("\n" + "="*70)
print("Overlap between Chunk 1 and Chunk 2:")
c1_end = overlap_chunks[0][-60:]  # Last 60 chars of chunk 1
c2_start = overlap_chunks[1][:60] # First 60 chars of chunk 2
print(f"  End of Chunk 1:   ...{c1_end!r}")
print(f"  Start of Chunk 2: {c2_start!r}...")

### Strategy 3: Section-Based Chunking

In [ ]:
def chunk_by_section(text, max_chunk_size=600):
    """
    Split text at natural section boundaries (double newlines / paragraph breaks).
    - Merges short adjacent sections up to max_chunk_size.
    - Splits overly long sections at sentence boundaries.
    """
    # Split on 2+ consecutive newlines
    raw_sections = re.split(r'\n{2,}', text)
    
    chunks = []
    current_chunk = ""
    
    for section in raw_sections:
        section = section.strip()
        if not section:
            continue
        
        if len(current_chunk) + len(section) <= max_chunk_size:
            # Merge this section into the current chunk
            current_chunk += (" " if current_chunk else "") + section
        else:
            # Save current chunk
            if current_chunk:
                chunks.append(current_chunk)
            
            # If this section alone is too big, split at sentence boundaries
            if len(section) > max_chunk_size:
                sentences = re.split(r'(?<=[.!?])\s+', section)
                temp = ""
                for sent in sentences:
                    if len(temp) + len(sent) <= max_chunk_size:
                        temp += (" " if temp else "") + sent
                    else:
                        if temp:
                            chunks.append(temp)
                        temp = sent
                current_chunk = temp
            else:
                current_chunk = section
    
    if current_chunk:
        chunks.append(current_chunk)
    
    return chunks


# Test on Document 2
section_chunks = chunk_by_section(DOCUMENTS[1], max_chunk_size=600)

print(f"Section-based chunking produced {len(section_chunks)} chunks from Doc 2")
print("\n" + "="*70)
for i, chunk in enumerate(section_chunks):
    print(f"\n[Chunk {i+1}] ({len(chunk)} chars):")
    print(chunk)
    print("-"*50)

### Chunking Strategy Comparison

In [ ]:
def compare_chunking_strategies(documents):
    """Apply all three strategies to the full corpus and compare statistics."""
    all_text = "\n\n".join(documents)
    
    strategies = {
        "Fixed-Size (300 chars)": chunk_fixed(all_text, chunk_size=300),
        "Overlapping (300/60)": chunk_overlap(all_text, chunk_size=300, overlap=60),
        "Section-Based (600 max)": chunk_by_section(all_text, max_chunk_size=600),
    }
    
    print(f"{'Strategy':<28} {'Chunks':>8} {'Avg Size':>10} {'Min':>6} {'Max':>6}")
    print("-" * 62)
    for name, chunks in strategies.items():
        sizes = [len(c) for c in chunks]
        print(f"{name:<28} {len(chunks):>8} {np.mean(sizes):>10.1f} {min(sizes):>6} {max(sizes):>6}")
    
    return strategies

chunk_strategies = compare_chunking_strategies(DOCUMENTS)

---
## Section 3: Loading the Embedding Model

We use `all-MiniLM-L6-v2` — a compact but powerful sentence transformer that produces **384-dimensional embeddings**. It's fast enough to run on CPU and performs well for semantic retrieval tasks.

**Important:** The same model must be used for both indexing and query encoding. If you index with one model and query with another, the vector spaces are incompatible and retrieval will fail silently.

In [ ]:
# Load embedding model (downloads ~22MB on first run)
print("Loading embedding model...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded.")

# Quick test: embed two semantically similar and one different sentence
test_sentences = [
    "Ibuprofen reduces fever and inflammation.",
    "NSAIDs are used to treat pain and high temperature.",
    "The stock market opened higher this morning."
]

test_embeddings = embed_model.encode(test_sentences)
print(f"\nEmbedding shape: {test_embeddings.shape}")

# Compute cosine similarities
def cosine_sim(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print(f"\nSimilarity (sentence 1 vs 2): {cosine_sim(test_embeddings[0], test_embeddings[1]):.4f}")
print(f"Similarity (sentence 1 vs 3): {cosine_sim(test_embeddings[0], test_embeddings[2]):.4f}")
print("\nSentences 1 and 2 should be much more similar than 1 and 3.")

---
## Section 4: Building the FAISS Vector Index

FAISS (Facebook AI Similarity Search) stores embeddings and answers nearest-neighbor queries in milliseconds — even over millions of vectors. We use `IndexFlatIP` (inner product) with L2-normalized vectors, which is equivalent to cosine similarity search.

In [ ]:
def build_faiss_index(chunks, model):
    """
    Embed all chunks and build a FAISS index.
    Returns the index and the normalized embeddings.
    """
    print(f"Embedding {len(chunks)} chunks...")
    embeddings = model.encode(chunks, show_progress_bar=True, batch_size=32)
    embeddings = embeddings.astype(np.float32)
    
    # Normalize for cosine similarity (inner product after normalization)
    faiss.normalize_L2(embeddings)
    
    # Build index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)
    
    print(f"Index built: {index.ntotal} vectors, {dimension} dimensions")
    return index, embeddings


# Build index using section-based chunks (all documents)
all_text = "\n\n".join(DOCUMENTS)
section_chunks_all = chunk_by_section(all_text, max_chunk_size=600)

print("Building index with section-based chunks...")
index, chunk_embeddings = build_faiss_index(section_chunks_all, embed_model)

---
## Section 5: Retrieval — Finding Relevant Chunks

The retrieval step:
1. Embeds the user's query using the same model
2. Performs similarity search in the FAISS index
3. Returns the top-k most relevant chunks with their similarity scores

In [ ]:
def retrieve(query, index, chunks, model, top_k=3):
    """
    Find the top_k most relevant chunks for the given query.
    Returns a list of dicts with chunk text, similarity score, and index.
    """
    # Embed and normalize the query
    query_emb = model.encode([query]).astype(np.float32)
    faiss.normalize_L2(query_emb)
    
    # Search
    scores, indices = index.search(query_emb, top_k)
    
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'chunk': chunks[idx],
            'score': round(float(score), 4),
            'index': int(idx)
        })
    return results


# Test retrieval with different queries
test_queries = [
    "What are the side effects of ibuprofen?",
    "What is the first-line treatment for hypertension?",
    "How often should HbA1c be measured in diabetes patients?",
    "When should antibiotics not be prescribed?"
]

for query in test_queries:
    print("\n" + "="*70)
    print(f"Query: {query}")
    print("-"*70)
    results = retrieve(query, index, section_chunks_all, embed_model, top_k=2)
    for i, r in enumerate(results):
        print(f"\n[Rank {i+1}] Score: {r['score']}")
        print(r['chunk'][:300] + ("..." if len(r['chunk']) > 300 else ""))

---
## Section 6: Building the Generator Prompt

The prompt is the interface between the retriever and the generator. A well-designed prompt:
- Clearly separates retrieved context from the user question
- Instructs the model to use ONLY the provided context
- Includes a fallback instruction for when context is insufficient

We use a stub generator here (no API key required). In production, replace `stub_generator` with a call to any LLM API.

In [ ]:
def build_prompt(query, retrieved_chunks):
    """
    Assemble a RAG prompt from retrieved chunks and the user query.
    """
    context_parts = []
    for i, r in enumerate(retrieved_chunks):
        context_parts.append(f"[Source {i+1} | Relevance: {r['score']}]\n{r['chunk']}")
    context = "\n\n---\n\n".join(context_parts)
    
    prompt = f"""You are a helpful clinical assistant. Answer the question ONLY based on the context provided below.
If the answer cannot be found in the context, respond with: "I don't know based on the provided documents."
Do not use any knowledge outside of the given context.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""
    return prompt


def stub_generator(prompt):
    """
    Stub generator — extracts and returns a summary of the retrieved context.
    In production, replace this with:
      response = openai.chat.completions.create(...)
      or
      response = anthropic.messages.create(...)
    """
    try:
        ctx_start = prompt.index("CONTEXT:") + len("CONTEXT:")
        ctx_end = prompt.index("QUESTION:")
        context = prompt[ctx_start:ctx_end].strip()
        # Remove source labels and take first ~200 chars
        clean = re.sub(r'\[Source \d+.*?\]\n', '', context)
        clean = re.sub(r'---', '', clean).strip()
        sentences = re.split(r'(?<=[.!?])\s+', clean)
        answer = ' '.join(sentences[:3])
        return answer.strip()
    except ValueError:
        return "Unable to generate answer."


# Full pipeline demo
demo_query = "What are the side effects of ibuprofen?"
print(f"Query: {demo_query}")
print("\n" + "="*70)

retrieved = retrieve(demo_query, index, section_chunks_all, embed_model, top_k=3)
prompt = build_prompt(demo_query, retrieved)
answer = stub_generator(prompt)

print("RETRIEVED CHUNKS:")
for r in retrieved:
    print(f"  Score {r['score']}: {r['chunk'][:80]}...")

print(f"\nGENERATED ANSWER:\n{answer}")

---
## Section 7: Full RAG Pipeline Class

We now package everything into a clean, reusable `SimpleRAG` class.

In [ ]:
class SimpleRAG:
    """
    A minimal, self-contained RAG pipeline.
    
    Phases:
      Offline: index_documents() — chunk → embed → FAISS index
      Online:  query()           — embed query → retrieve → build prompt → generate
    """

    def __init__(self, model_name='all-MiniLM-L6-v2'):
        self.model = SentenceTransformer(model_name)
        self.chunks = []
        self.index = None
    
    # ── OFFLINE PHASE ─────────────────────────────────────────────────
    def index_documents(self, documents, chunk_fn, **chunk_kwargs):
        """Chunk all documents and build the FAISS index."""
        self.chunks = []
        for doc in documents:
            self.chunks.extend(chunk_fn(doc, **chunk_kwargs))
        
        print(f"Chunked into {len(self.chunks)} chunks.")
        
        embeddings = self.model.encode(self.chunks, show_progress_bar=False)
        embeddings = embeddings.astype(np.float32)
        faiss.normalize_L2(embeddings)
        
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)
        self.index.add(embeddings)
        print(f"Index built: {self.index.ntotal} vectors ({dim}D).")
    
    # ── ONLINE PHASE ──────────────────────────────────────────────────
    def retrieve(self, query, top_k=3):
        """Return top_k relevant chunks for the query."""
        q = self.model.encode([query]).astype(np.float32)
        faiss.normalize_L2(q)
        scores, indices = self.index.search(q, top_k)
        return [
            {'chunk': self.chunks[idx], 'score': round(float(s), 4), 'index': int(idx)}
            for s, idx in zip(scores[0], indices[0])
        ]
    
    def build_prompt(self, query, retrieved):
        """Assemble a grounded prompt from retrieved chunks."""
        context = "\n\n---\n\n".join(
            [f"[Source {i+1}]\n{r['chunk']}" for i, r in enumerate(retrieved)]
        )
        return (
            "Answer ONLY using the provided context. "
            "If the answer is not in the context, say 'I don't know based on the provided documents.'\n\n"
            f"CONTEXT:\n{context}\n\nQUESTION: {query}\n\nANSWER:"
        )
    
    def query(self, question, generator_fn=stub_generator, top_k=3):
        """Full pipeline: retrieve → prompt → generate."""
        retrieved = self.retrieve(question, top_k)
        prompt = self.build_prompt(question, retrieved)
        answer = generator_fn(prompt)
        return {
            'question': question,
            'retrieved': retrieved,
            'answer': answer
        }


# Instantiate and test
rag = SimpleRAG()
rag.index_documents(DOCUMENTS, chunk_by_section, max_chunk_size=600)

result = rag.query("What is the maximum daily dose of ibuprofen for over-the-counter use?")
print(f"\nQuestion: {result['question']}")
print(f"Answer: {result['answer']}")

---
## Section 8: Grounding Check

A grounding check answers the question: **does the generated answer actually reference the retrieved context?**

We implement two complementary approaches:
- **Keyword overlap** — fast, interpretable, token-level
- **Semantic similarity** — slower, more robust, meaning-level

In [ ]:
STOPWORDS = {
    'the','a','an','is','are','was','were','in','of','to','and','or',
    'for','with','that','this','it','be','as','on','at','by','from',
    'which','not','no','its','their','have','has','been','but','if',
    'should','can','may','will','would','could','such','any','also'
}

def tokenize(text):
    return [w.lower() for w in re.findall(r'\b[a-zA-Z]{3,}\b', text)
            if w.lower() not in STOPWORDS]


def grounding_keyword(answer, retrieved_chunks):
    """
    Fraction of content words in the answer that appear in the retrieved context.
    Score range: 0.0 (not grounded) → 1.0 (fully grounded).
    """
    answer_words = set(tokenize(answer))
    context_text = " ".join([r['chunk'] for r in retrieved_chunks])
    context_words = set(tokenize(context_text))
    
    if not answer_words:
        return 0.0
    
    overlap = answer_words & context_words
    return round(len(overlap) / len(answer_words), 3)


def grounding_semantic(answer, retrieved_chunks, model):
    """
    Maximum cosine similarity between the answer and any retrieved chunk.
    Score range: 0.0 (semantically unrelated) → 1.0 (semantically identical).
    """
    answer_emb = model.encode([answer]).astype(np.float32)
    chunk_embs = model.encode([r['chunk'] for r in retrieved_chunks]).astype(np.float32)
    
    faiss.normalize_L2(answer_emb)
    faiss.normalize_L2(chunk_embs)
    
    similarities = (chunk_embs @ answer_emb.T).squeeze()
    return round(float(np.max(similarities)), 3)


def grounding_report(answer, retrieved_chunks, model):
    """Print a full grounding report for a QA result."""
    kw_score = grounding_keyword(answer, retrieved_chunks)
    sem_score = grounding_semantic(answer, retrieved_chunks, model)
    
    print(f"  Keyword overlap score:  {kw_score}")
    print(f"  Semantic similarity:    {sem_score}")
    
    overall = (kw_score + sem_score) / 2
    if overall >= 0.6:
        verdict = "✅ Well grounded"
    elif overall >= 0.35:
        verdict = "⚠️  Partially grounded"
    else:
        verdict = "❌ Not grounded — possible hallucination"
    
    print(f"  Combined score:         {overall:.3f}  →  {verdict}")
    return kw_score, sem_score


# Run grounding check on several queries
eval_queries = [
    "What are the side effects of ibuprofen?",
    "What is the first-line treatment for hypertension?",
    "How is HbA1c used in diabetes management?",
    "What is the capital of France?"  # Out-of-domain query — should score low
]

for q in eval_queries:
    print("\n" + "="*70)
    print(f"Query: {q}")
    res = rag.query(q)
    print(f"Answer: {res['answer'][:200]}")
    print("Grounding check:")
    grounding_report(res['answer'], res['retrieved'], rag.model)

---
## Section 9: Comparing Chunking Strategies on Retrieval Quality

Now we run a structured experiment: build three RAG systems with different chunking strategies, send the same queries to each, and compare retrieval quality using grounding scores.

In [ ]:
eval_queries = [
    "What are the side effects of ibuprofen?",
    "What is the first-line treatment for hypertension?",
    "When should antibiotics not be prescribed?",
    "What is the target HbA1c level for diabetes?"
]

strategies_config = [
    ("Fixed-Size",    chunk_fixed,       {"chunk_size": 300}),
    ("Overlapping",   chunk_overlap,     {"chunk_size": 300, "overlap": 60}),
    ("Section-Based", chunk_by_section,  {"max_chunk_size": 600}),
]

results_table = {name: [] for name, _, _ in strategies_config}

for strat_name, chunk_fn, kwargs in strategies_config:
    print(f"\n{'='*70}")
    print(f"Strategy: {strat_name}")
    print(f"{'='*70}")
    
    rag_s = SimpleRAG()
    rag_s.index_documents(DOCUMENTS, chunk_fn, **kwargs)
    
    for q in eval_queries:
        res = rag_s.query(q)
        kw, sem = grounding_keyword(res['answer'], res['retrieved']), \
                  grounding_semantic(res['answer'], res['retrieved'], rag_s.model)
        combined = round((kw + sem) / 2, 3)
        results_table[strat_name].append(combined)
        print(f"  Q: {q[:50]:<50}  Score: {combined}")

# Summary
print("\n" + "="*70)
print("GROUNDING SCORE SUMMARY (higher = better grounded)")
print("="*70)
print(f"{'Strategy':<20} {'Avg Score':>12}")
print("-"*35)
for name, scores in results_table.items():
    print(f"{name:<20} {np.mean(scores):>12.3f}")

---
## Section 10: Summary and Exercises

### What We Built

| Component | Implementation |
|---|---|
| Document corpus | 4 synthetic clinical knowledge base articles |
| Chunking — Fixed | `chunk_fixed()` — character-level splits |
| Chunking — Overlap | `chunk_overlap()` — sliding window with shared content |
| Chunking — Section | `chunk_by_section()` — semantic paragraph boundaries |
| Embedding model | `all-MiniLM-L6-v2` (384D sentence embeddings) |
| Vector store | FAISS `IndexFlatIP` with L2-normalized vectors |
| Generator | Stub (replace with any LLM API for production) |
| Grounding check | Keyword overlap + semantic cosine similarity |

### Key Insights

1. **Chunking strategy affects retrieval precision.** Section-based chunking preserves semantic units; fixed-size may truncate key information mid-sentence.

2. **Grounding scores help diagnose failures.** A low grounding score on an in-domain query suggests a retrieval failure or a mismatch between chunks and query intent.

3. **The same embedding model must be used throughout.** Mixing models at indexing and query time produces meaningless similarity scores.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# EXERCISES — Complete these to test your understanding
# ─────────────────────────────────────────────────────────────────

# Exercise 1: Add a new document to DOCUMENTS about asthma management.
# Re-index the RAG system and test queries like:
#   "What is the first-line treatment for asthma?"
# Verify that the answer references the new document.

# YOUR CODE HERE


# Exercise 2: Modify chunk_by_section() to also split at lines
# that look like headings (e.g., lines with no punctuation, or all-caps lines).
# Test it on Document 1 (Hypertension) and compare chunk count.

# YOUR CODE HERE


# Exercise 3: Implement a grounding check that uses the RETRIEVAL SCORE
# (from FAISS) as a proxy for grounding. If the top-1 retrieved chunk
# has a score below 0.5, flag the answer as "low confidence".

# YOUR CODE HERE


# Exercise 4 (Challenge): Implement a simple re-ranking step.
# After retrieving top-5 chunks, re-rank them by keyword overlap with
# the query. Return only the top-3 after re-ranking.
# Does this improve grounding scores on the eval queries?

# YOUR CODE HERE

print("Exercises ready — begin coding!")